In [1]:
# Simple module imports
import numpy as np
import matplotlib.pyplot as plt
import json
import os
# from numpy.polynomial import polynomial
# from scipy.stats import norm

# Lazy imports - comment out when not in use
from constants import *
from source.utilities import *
from source.config import *
# from analysis.data_plotting import *
# from analysis.critical_exponent import *
from QSHE.testing_qshe import *
from Local.run_local_qshe import *


rng = build_rng(1234)

### Helper functions for the notebook

In [2]:
# Setup vars
from collections import defaultdict
theta_vals = range(1, 7)
thetas = {"1": 0, "2": r"$\pi/8$", "3": r"$3/pi/16$", "4": r"$/pi/4$", "5": r"$3/pi/8$", "6": r"$\pi/2$"}

def find_intersections(xvals, ydict, totalsteps, metrictype = "mean"):
    """
    Given an array of x values and multiple y-value arrays (over the same x-axis), obtain the (x,y) coordinate where all lines intersect.
    Due to discretisation error, we set a tolerance for 'intersection' in y
    We compute the mean y value per x value, find the variance for all steps, and see whether there exists points where the variance of all curves falls below our tolerance
    """
    # We expect the ydict to be of the form ydict[xval][step] = (mean, median)
    # And the metrictype will tell us which value to use for checking
    variances = {}
    for xval in xvals:
        # First we obtain all the yvalues for a particular xval
        yvals = []
        for step in range(totalsteps):
            ymean, ymedian = ydict[xval][step][0]
            if metrictype == "mean":
                yvals.append(ymean)
            elif metrictype == "median":
                yvals.append(ymedian)
            else:
                raise ValueError(f"{metrictype} is an invalid or inaccurate metric to use. Use 'mean' or 'median'.")
        yarray = np.array(yvals)
        # Then we find its mean and variance
        yarray_var = np.var(yarray)
        variances.update({xval:yarray_var})
    return variances


def close_old_plots():
    """Close all old open figures"""
    openfigs = plt.get_fignums()
    for fignum in openfigs:
        plt.close(fignum)

def convert_z_to_x(z):
    return np.arcsinh(np.exp(z/2))

def convert_g_to_x(g, theta=0.0):
    z = convert_g_to_z(g)
    return convert_z_to_x(z)

def convert_x_to_z(x):
    return np.log(np.sinh(x)**2)

def convert_x_to_g(x):
    z = convert_x_to_z(x)
    return convert_z_to_g(z)

def load_var_moments(rg_steps, var, val):
    # Load prev files with varying thetas
    filename = f"{local_dir}/archived data/theta test/theta_test_{val}_numerical_shreyas/FP/stats/{var}_moments.json"
    moments = []
    with open(filename, "r") as file:
        data = json.load(file)
    for step in range(rg_steps):
        try:
            moments.append(data[f"RG_{step}"]["mean"])
        except KeyError:
            continue
    return moments

def get_median_and_mode(rg_steps, var, version, thetanum):
    """Compute the median and mode of histogram data"""
    foldername = f"{local_dir}/theta_{thetanum}/{version}/FP/hist/{var}"
    medians = []
    modes = []
    for step in range(rg_steps):
        try:
            filename = f"{foldername}/{var}_hist_RG{step}.npz"
            data = np.load(filename)
        except FileNotFoundError:
            continue
        medians.append(np.median(data["histval"]))
        mode_index = np.argmax(data["histval"])
        mode = data["bincenters"][mode_index]
        modes.append(mode)
    return medians, modes


def load_fixed_theta_moments(rg_steps, fixedvar, thetanum, fixedvarval, var):
    """Load moments from old trials at a fixed theta"""
    filename = f"{local_dir}/theta_{thetanum}/theta_{thetanum}_{fixedvar}{fixedvarval}_numerical_shreyas/FP/stats/{var}_moments.json"
    moments = []
    with open(filename, "r") as file:
        data = json.load(file)
    for step in range(rg_steps):
        try:
            moments.append(data[f"RG_{step}"]["mean"])
        except KeyError:
            continue
    return moments

def load_hists(rg_steps, fixedvar, thetanum, fixedvarval, var):
    """Load histograms from theta trials"""
    filename = f"{local_dir}/theta_{thetanum}_{fixedvar}{fixedvarval}_numerical_shreyas/FP/hist/{var}/{var}_hist"
    histdata = []
    for step in range(rg_steps):
        try:
            histfile = f"{filename}_RG{step}.npz"
            ht = np.load(histfile)
            histdata.append(ht)
        except KeyError or FileNotFoundError:
            continue
    return histdata

def collapse_data(data_dict):
    """Collapse nested dictionaries into a json saveable format"""
    if isinstance(data_dict, dict):
        return {collapse_data(key): collapse_data(value) for key, value in data_dict.items()}
    elif isinstance(data_dict, np.ndarray):
        return data_dict.tolist()
    else:
        return data_dict

def save_metric_json(data, filename):
    """Save input data into a json file"""
    with open(filename, "w") as f:
        json.dump(data, f, indent=2)

def build_state_dict(qvals, gvals, nsamples, steps, metric, fix=True):
    """Build a state dict containing relevant config params"""
    state = {}
    num_qs = int(len(qvals))
    num_gs = int(len(gvals))
    min_q = float(min(qvals))
    max_q = float(max(qvals))
    min_g = float(min(gvals))
    max_g = float(max(gvals))
    trimmed_qs = [round(float(q), 3) for q in qvals]
    trimmed_gs = [round(float(g), 3) for g in gvals]
    state.update({"q":{"Num": num_qs, "Max": round(max_q, 3), "Min": round(min_q, 3), "Data": trimmed_qs}})
    state.update({"g":{"Num": num_gs, "Max": round(max_g, 3), "Min": round(min_g, 3), "Data": trimmed_gs}})
    state.update({"data": {"type": metric, "samples": nsamples, "steps": steps, "fixed": fix}})
    return state

def quicker_trials(qs, phis, g_init, samples, nsteps, metric = "mean", var = "g", fix = True):
    gmeans = defaultdict(list)
    qmeds = defaultdict(list)
    for q in qs:
        t_init = rng.uniform(np.sqrt(g_init-1e-5), np.sqrt(g_init+1e-5), samples)
        f_init = q*(1 - t_init**2)
        # print(np.mean(t_init), np.mean(f_init), g_init)
        for step in range(nsteps):
            indices=  rng.integers(0, samples, (samples, 5))
            ts = np.take(t_init, indices)
            fs = np.take(f_init, indices)
            tp = numerical_solver(ts, fs, phis, samples, 2, [1.0,0.0,0.0,0.0], samples)
            tp = np.clip(tp, 1e-6, 1.0-1e-6)
            p = tp**2
            if not fix:
                fp = numerical_solver(ts, fs, phis, samples, 17, [1.0,0.0,0.0,0.0], samples)
            else:
                fp = np.sqrt((1-p)*q)
            # fp = np.clip(fp, 1e-6, 1.0-1e-6)
            f2 = fp**2
            g = p + f2
            tmax = np.max(tp)
            tmin = np.min(tp)
            fmax = np.max(fp)
            fmin = np.min(fp)
            if tmax > 1.0 or fmax > 1.0 or np.max(f2) > 1.0:
                print(f"tmax = {tmax}, fmax = {fmax}. At step {step} for ginit = {g_init} and q = {q}")
            if tmin < 0.0 or fmin < 0.0 or np.min(f2) < 0.0:
                print(f"tmax = {tmax}, fmax = {fmax}. At step {step} for ginit = {g_init} and q = {q}")
            # g = np.clip(g, 1e-12, 1.0-1e-12)
            # tp= np.clip(tp, 1e-12, 1-1e-12)
            # fp = np.clip(fp, 1e-12, 1-1e12)
            if var == "z":
                gp = convert_g_to_z(g)
                # gp = np.log((1 - g - f2)/g)
            elif var == "x":
                gp = convert_g_to_x(g)
            elif var == "p":
                gp = p
            else:
                gp = g
            unique, counts = np.unique(gp, return_counts=True)
            maxunique = np.argmax(counts)
            mode = unique[maxunique]
            median = np.median(gp)
            mean = np.mean(gp)
            qmed = np.median(f2/(1-p))
            qmean = np.mean(f2/(1-p))
            qmeds[step].append((qmean, qmed))
            if metric == "mode":
                gmeans[step].append(mode)
            elif metric == "median":
                gmeans[step].append(median)
            elif metric == "all":
                gmeans[step].append((mean, median))
            else:
                gmeans[step].append(mean)
            t_init = tp
            f_init = fp
    # print(q, qmeds)
    return gmeans, qmeds

In [3]:
markers = [",", "o", "v", "^", "1", "2", "3", "4", "8", "s", "p", "P", "*", "+"]
# Load data for the desired vars
z_moments = defaultdict(list)
g_moments = defaultdict(list)
for val in theta_vals:
    z_moments[f"{val}"] = load_var_moments(12, "z", val)
    g_moments[f"{val}"] = load_var_moments(12, "g", val)


In [4]:
import matplotlib.colors as mcolors
loaded_colors = mcolors.TABLEAU_COLORS
avail_colors = len(loaded_colors)
avail_markers = len(markers)

In [5]:
close_old_plots() # Close old plots for a quick reset when desired

### Inspect means for data with varying thetas

In [ ]:
# Create figures and store for later
plt.ioff()
fz, axz = plt.subplots(num="fz")
xaxis = range(1,13)
axz.axhline(y=0.0, linestyle="--", color="k", alpha=0.6)
xmeans = {}

for key in z_moments.keys():
    num = len(z_moments[key])
    xmeans.update({key: convert_z_to_x(np.array(z_moments[key]))})
    marker=rng.integers(0,len(markers), 1, dtype=int)
    a=axz.scatter(xaxis[:num], z_moments[key], label=thetas[key], marker=markers[marker[0]])
    c=a.get_facecolor()[0]
    axz.scatter(0, 0.0, color=c, marker=markers[marker[0]])
axz.set_xlabel("RG step")
axz.set_ylabel(r"$z_{\text{mean}}$")
axz.set_title(r"$z_{\text{mean}}$ vs RG steps for various $\theta$")
axz.legend(loc="upper right", bbox_to_anchor=(1.2, 1.0))

fx, axx = plt.subplots(num="fx")
axx.axhline(y=np.log(1 + np.sqrt(2)), linestyle="--", color="k", label=r"x=ln$(1+\sqrt{2})$", alpha=0.5)

for key in xmeans.keys():
    num = len(xmeans[key])
    marker=rng.integers(0,len(markers), 1, dtype=int)
    a=axx.scatter(xaxis[:num], xmeans[key], label=thetas[key], marker=markers[marker[0]])
    c=a.get_facecolor()[0]
    axx.scatter(0.0, np.log(1+np.sqrt(2)), color=c, marker=markers[marker[0]])

axx.set_xlabel("RG step")
axx.set_ylabel("x")
axx.set_title(r"x vs RG steps for various $\theta$")
axx.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))

fg, axg = plt.subplots(num="fg")
axg.axhline(y=0.5, linestyle="--", color="k", label="g=0.5", alpha=0.5)

for key in g_moments.keys():
    num = len(g_moments[key])
    marker=rng.integers(0,len(markers), 1, dtype=int)
    a=axg.scatter(xaxis[:num], g_moments[key], label=thetas[key], marker=markers[marker[0]])
    c=a.get_facecolor()[0]
    axg.scatter(0.0,0.5, color=c, marker=markers[marker[0]])
axg.set_xlabel("RG step")
axg.set_ylabel(r"$g_{\text{mean}}$")
axg.set_title(r"$g_{\text{mean}}$ vs RG steps for various $\theta$")
axg.legend(loc="upper right", bbox_to_anchor=(1.2, 1.0))


### Inspect data for varying g, fixed $\theta = \pi/8$

In [ ]:
markers = [",", "o", "v", "^", "1", "2", "3", "4", "8", "s", "p", "P", "*", "+"]
colors = []
# Now we see the data for varying gs
gvals = [0.1, 0.2, 0.6, 0.7, 0.9]
g_hists = defaultdict(list)
z_mom = defaultdict(list)
g_mom = defaultdict(list)
z_med = defaultdict(list)
z_mode = defaultdict(list)
g_med = defaultdict(list)
g_mode = defaultdict(list)
mcopy = markers.copy()
pres_vals = []
for gval in gvals:
    z_mom[f"{gval}"] = load_fixed_theta_moments(12, "g", 2, gval, "z")
    g_mom[f"{gval}"] = load_fixed_theta_moments(12, "g", 2, gval, "g")
    z_med[f"{gval}"], z_mode[f"{gval}"] = get_median_and_mode(12, "z", f"theta_2_g{gval}_numerical_shreyas", 2)
    g_med[f"{gval}"], g_mode[f"{gval}"] = get_median_and_mode(12, "g", f"theta_2_g{gval}_numerical_shreyas", 2)
    try:
        g_hists[f"{gval}"] = load_hists(12, "g", 2, gval, "g")
        pres_vals.append(gval)
    except FileNotFoundError:
        continue
glabs = gvals
glabs.insert(2,0.5)
m = rng.integers(0, len(markers), len(glabs))
mcopy = np.array(markers)[m]
vth = list(THETA_DICT.keys())[1:-1]
g09th5 = load_fixed_theta_moments(12, "g", 5, 0.9, "g")
# print(g09th5)
# print(g_moments['1'])
# print(g_mom.keys())

In [ ]:
for val in gvals:
    plt.scatter(range(len(z_mode[f"{val}"])),z_mode[f"{val}"], label=f"{val}")
plt.legend()
# print(z_mode)

In [ ]:
# plt.ioff()
ftz, axtz = plt.subplots(num="ftz")
glabel = r"$g_{\text{init}}$"
# xaxis = range(12)
axtz.axhline(y=0.0, linestyle="--", color="k", alpha=0.6, label="z=0")
axtz.axhline(y=25.0, linestyle="--", color="r", alpha=0.6, label="z=25")
xtmeans = {}
i = 0
zm = np.array([float(k) for k in z_mom.keys()])
xinit = convert_z_to_x(zm)
for key in z_mom.keys():
    num = len(z_mom[key])
    xtmeans.update({key: convert_z_to_x(np.array(z_mom[key]))})
    marker=rng.integers(0,len(markers), 1, dtype=int)
    a =axtz.scatter(xaxis[:num], z_mom[key], label=f"{glabel}={key}",marker=markers[marker[0]])
    c = a.get_facecolor()[0]
    axtz.scatter(0.0, convert_g_to_z(np.array(float(key))), color=c,marker=markers[marker[0]])
axtz.set_xlabel("RG step")
axtz.set_ylabel(r"$z_{\text{mean}}$")
axtz.set_title(r"$z_{\text{mean}}$ with $\theta = \pi/8$ vs RG steps for various $g_{init}$")
axtz.legend(loc="upper right", bbox_to_anchor=(1.25, 1.0))

ftx, axtx = plt.subplots(num="ftx")
axtx.axhline(y=np.log(1 + np.sqrt(2)), linestyle="--", color="k", label=r"x=ln$(1+\sqrt{2})$", alpha=0.5)
i=0
for key in xtmeans.keys():
    num = len(xtmeans[key])
    marker=rng.integers(0,len(markers), 1, dtype=int)
    a = axtx.scatter(xaxis[:num], xtmeans[key], label=f"{glabel}={key}",marker=markers[marker[0]])
    c = a.get_facecolor()[0]
    axtx.scatter(0.0, xinit[i], color=c,marker=markers[marker[0]])
    i+=1
axtx.set_xlabel("RG step")
axtx.set_ylabel("x")
axtx.set_title(r"x vs RG steps with $\theta = \pi/8$ for various $g_{init}$")
axtx.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))

ftg, axtg = plt.subplots(num="ftg")
axtg.axhline(y=0.5, linestyle="--", color="k", label="g=0.5", alpha=0.5)
# axtg.scatter(xaxis)
for key in g_mom.keys():
    num = len(g_mom[key])
    marker=rng.integers(0,len(markers), 1, dtype=int)
    a = axtg.scatter(xaxis[:num], g_mom[key], label=f"{glabel}={key}",marker=markers[marker[0]])
    c = a.get_facecolor()[0]
    axtg.scatter(0.0, float(key), color=c,marker=markers[marker[0]])
axtg.set_xlabel("RG step")
axtg.set_ylabel(r"$g_{\text{mean}}$")
axtg.set_title(r"$g_{\text{mean}}$ with $\theta = \pi/8$ vs RG steps for various $g_{init}$")
axtg.legend(loc="upper right", bbox_to_anchor=(1.25, 1.0))

### Print plots for all 3 vars and desired number of steps

In [ ]:
# Helper function
def theta_var_plot(s: int, var: str, save: bool = False):
    moments = defaultdict(list)
    mom = defaultdict(list)
    # mainval = 0.5
    if var == "z":
        labs = convert_g_to_z(np.array(glabs))
        mainval = convert_g_to_z(np.array(0.5))
        ninevar = convert_g_to_z(np.array(0.9))
        for key in g_moments.keys():
            moments[key] = list(convert_g_to_z(np.array(g_moments[key])))
        for key in g_mom.keys():
            mom[key] = list(convert_g_to_z(np.array(g_mom[key])))
        var09th5 = list(convert_g_to_z(np.array(g09th5)))
    elif var == "x":
        labs = convert_g_to_x(np.array(glabs))
        mainval = convert_g_to_x(np.array(0.5))
        ninevar = convert_g_to_x(np.array(0.9))
        for key in g_moments.keys():
            moments[key] = list(convert_g_to_x(np.array(g_moments[key])))
        for key in g_mom.keys():
            mom[key] = list(convert_g_to_x(np.array(g_mom[key])))
        var09th5 = list(convert_g_to_x(np.array(g09th5)))
    else:
        labs = glabs
        mainval = 0.5
        ninevar = 0.9
        var09th5 = g09th5
        moments = g_moments.copy()
        mom = g_mom.copy()

    # print(moments)
    # print(mom)
    # print(var09th5)
    narr = np.zeros(shape=(len(labs), s+2))
    narr[:,0] = labs
    narr[:,1] = np.tile(mainval, 6).T
    sj = 0
    for key in moments.keys():
        narr[sj, 0] = THETA_DICT[key]
        # c = a.get_facecolor()[0]
        for i in range(s):
            narr[sj, i+2] = moments[key][i]
        sj += 1
    # print(narr)
    # pres_vals.append(0.5)
    plt.figure(figsize=(10,6))
    k=0
    for ind, gv in enumerate(glabs):
        if gv == 0.5:
            plt.scatter(narr[:, 2:], np.tile(narr[:,0], s).reshape(s, 6).T, color = "m", marker = mcopy[k], alpha=0.7)
            plt.scatter(narr[:, 1], narr[:,0], color="m", marker=mcopy[k], label=f"g_init = {gv}", s = 100, alpha=0.7)
            k+=1
            continue
        # steps = len(g_hists[f"{gv}"])
        a = plt.scatter(labs[ind], np.pi/8, label=f"g_init = {gv}", marker=mcopy[k], alpha=0.9, s=100)
        c = a.get_facecolor()[0]
        for i in range(s):
            try:
                plt.scatter(mom[f"{gv}"][i], np.pi/8, marker=mcopy[k], color=c)
            except IndexError:
                print(f"{var} = {labs[ind]:.3f} data does not have more than {i} steps. Proceeding with next")
                # continue

        if gv == ninevar:
            plt.scatter(labs[ind], THETA_DICT["5"], color=c, marker=mcopy[k], alpha=0.9, s = 100)
            plt.scatter(var09th5[:s], np.tile(THETA_DICT["5"], s), color=c, marker=mcopy[k])
        k+=1
        # plt.arrow(gv, np.pi/8, -(g_mom[f"{gv}"][-1]),0, color=c, head_width=0.03, alpha=0.5)


    plt.title(r"Plot of $\theta$ and {var} over {s} RG steps".format(var=var, s=s))
    plt.xlabel(f"{var}")
    thsa = list((THETA_DICT.keys()))
    ls = [thetas[l] for l in thsa if l in thetas.keys()]
    plt.xticks(labs)
    plt.yticks(list(THETA_DICT.values())[1:-2],ls)
    plt.grid(True, alpha=0.8)
    plt.ylabel(r"$\theta$")
    plt.legend(loc="upper right", bbox_to_anchor=(1.2, 1.0))
    if var == "z":
        plt.xlim((-3.0, 3.0))
    elif var == "x":
        plt.xlim((0.0, 2.0))
    # plt.show()
    if save:
        varfolder = f"{qshe_test_dir}/Theta_{var}"
        os.makedirs(varfolder, exist_ok=True)
        plt.savefig(f"{varfolder}/theta_{var}_{s}steps.png", dpi=150, bbox_inches="tight")
    # np.histogram2d()

In [ ]:
# close_old_plots()
# vars = ["z", "x", "g"]
# steps = 13
# for var in vars:
#     for step in range(steps):
#         close_old_plots()
#         theta_var_plot(step, var)

In [ ]:
close_old_plots() # Extra functionality if required
# Display previously generated figures
from IPython.display import display

# display(fz)
# display(fg)
# display(fx)
# display(ftz)
# display(ftx)
# display(ftg)

### Some new checks

In [ ]:
def quick_trials(thetas, phis, g_init, samples, nsteps, metric = "mean", var = "g"):
    gmeans = defaultdict(list)
    for theta in thetas:
        costheta = np.cos(theta)**2
        sintheta = np.sin(theta)**2
        t_init = np.full((samples), fill_value = np.sqrt(costheta*g_init))
        f_init = np.full((samples), fill_value = np.sqrt(sintheta*g_init))
        for step in range(nsteps):
            indices = rng.integers(0, samples, (samples, 5))
            ts = np.take(t_init, indices)
            fs = np.take(f_init, indices)
            tp = numerical_solver(ts, fs, phis, samples, 2, [1.0,0.0,0.0,0.0], samples)
            fp = numerical_solver(ts, fs, phis, samples, 17, [1.0,0.0,0.0,0.0], samples)
            gp = tp**2 + fp**2
            if var == "z":
                gp = convert_g_to_z(gp) # This is z, named gp just for ease of use
            elif var == "x":
                gp = convert_g_to_x(gp) # This is x, named gp just for ease of use
            if metric == "mode":
                unique, counts = np.unique(gp, return_counts=True)
                maxunique = np.argmax(counts)
                mode = unique[maxunique]
                gmeans[step].append(mode)
            elif metric == "median":
                median = np.median(gp)
                gmeans[step].append(median)
            elif metric == "all":
                unique, counts = np.unique(gp, return_counts=True)
                maxunique = np.argmax(counts)
                mode = unique[maxunique]
                median = np.median(gp)
                gmean = np.mean(gp)
                gmeans[step].append((gmean, median, mode))
            else:
                gmean = np.mean(gp)
                gmeans[step].append(gmean)
            t_init = tp
            f_init = fp
        # print(f"Finished data gen for θ = {theta}")
    return gmeans

In [ ]:
ntheta = 11
num_g = 10
lower_g = 1e-6
upper_g = 0.7
n = 50000
nsteps = 2
c_indices = rng.integers(0, avail_colors, num_g)
m_indices = rng.integers(0, avail_markers, num_g)
colors = np.take(list(loaded_colors.keys()), c_indices)
marks = np.take(markers, m_indices)


### Sweep through $\theta \in [0, \pi/2]$ for each $\text{g}_{init} \in [0, 1]$

In [ ]:
thetas = np.linspace(0, np.pi/2, ntheta)
g_inits = np.linspace(lower_g, upper_g, num_g)
gs = {g_init: defaultdict(list) for g_init in g_inits}
for i,ginit in enumerate(g_inits):
    phis = generate_random_phases(n, rng, 16)
    gs[ginit] = quick_trials(thetas, phis, ginit, nsamples, nsteps)
    if i % 3 == 0:
        print(f"Finished data gen for g_init = {ginit}")


In [ ]:
view_step = 1
plt.figure(figsize=(12, 8))
min_g = 0
max_g = 9
xvals = convert_g_to_x(g_inits[min_g:max_g])
for i, ginit in enumerate(g_inits[min_g:max_g]):
    plt.axvline(x=ginit, linestyle="--", color = colors[i+1], label=f"g_init = {ginit:.3f}", alpha=0.8)
    plt.scatter(gs[ginit][view_step],thetas,  color=colors[i+1], marker=marks[i])
    plt.fill_betweenx(thetas, ginit, gs[ginit][view_step], color=colors[i+1], alpha=0.3)

# plt.axvline(0.0, linestyle="--", alpha=0.4, label=r"$\theta = 0$", color="m")
# plt.axvline(np.pi, linestyle="--", alpha=0.4, label=r"$\theta=\pi$", color="m")
plt.ylabel(r"$2\theta/\pi$")
plt.xlabel("g")
# plt.xlim((min_g/1.2, max_g*1.2))
plt.title(f"Plot of g_mean after {view_step+1} RG step for {ntheta} thetas and {nsamples} samples")
# plt.yticks(g_inits)
plt.yticks(thetas[::2], labels=[str(t) for t in thetas[::2]/(np.pi)])
# plt.xticks(g_inits[min_g:max_g])
# plt.xlim((g_inits[max(min_g-1, 0)], g_inits[min(max_g+1, num_g-1)]))
plt.legend(loc="upper right", bbox_to_anchor=(1.2, 1.0))
plt.grid(alpha=0.5)
# plt.show()
os.makedirs(f"{qshe_test_dir}/Theta_G/{nsamples}samples/{view_step+1}steps", exist_ok=True)
plt.savefig(f"{qshe_test_dir}/Theta_G/{nsamples}samples/{view_step+1}steps/theta_g{g_inits[min_g]:.2f}-{g_inits[max_g]:.2f}_{ntheta}θ_{nsamples}samples_{view_step+1}steps.png", dpi=150, bbox_inches="tight")

### Sweep through $\text{g}_{init} \in [0, 1]$ with constant $\theta$

In [ ]:
ntheta = 1
num_g = 30
lower_g = 0.1
upper_g = 0.9
nsamples = 10000
nsteps = 6
c_indices = rng.integers(0, avail_colors, num_g)
m_indices = rng.integers(0, avail_markers, num_g)
colors = np.take(list(loaded_colors.keys()), c_indices)
marks = np.take(markers, m_indices)
var = "g"

In [ ]:
theta_fixed = np.full(ntheta, fill_value = 0)
g_s = np.linspace(lower_g, upper_g, num_g)
gms = {g_init: defaultdict(list) for g_init in g_s}
for i,ginit in enumerate(g_s):
    phis = generate_random_phases(nsamples, rng, 16)
    gms[ginit] = quick_trials(theta_fixed, phis, ginit, nsamples, nsteps, "all", var)
    if i % 5 == 0:
        print(f"Finished data gen for g_init = {ginit}")
print("Finished data gen")

In [ ]:
# print(gms[g_s[0]])
# print(convert_g_to_z(g_s[0:20]))
var = "z"


In [ ]:
view_step = 0
# plt.figure(figsize=(12, 8))
fig, axs = plt.subplots(1, 3, figsize=(24, 8))
min_g = 0
max_g = 30
minmean,minmed,minmode = 0,0,0
maxmean,maxmed,maxmode = 0,0,0
if var == "z":
    # gval_s  = convert_g_to_z(g_s)
    gval_s = np.log(g_s)
elif var == "x":
    gval_s = convert_g_to_x(g_s, theta_fixed[0])
else:
    gval_s = g_s
for step in range(nsteps):
    gmeans = []
    gmedians = []
    gmodes = []
    for i, ginit in enumerate(g_s[min_g:max_g]):
        gmean, gmedian, gmode = gms[ginit][step][0]
        if var == "x":
            gmean = convert_g_to_x(gmean, theta_fixed[0])
            gmedian = convert_g_to_x(gmedian, theta_fixed[0])
            gmode = convert_g_to_x(gmode, theta_fixed[0])
        elif var == "z":
            gmean = convert_g_to_z(gmean)
            gmedian = convert_g_to_z(gmedian)
            gmode = convert_g_to_z(gmode)
        gmeans.append(gmean)
        gmedians.append(gmedian)
        gmodes.append(gmode)
        minmean = min(minmean, gmean)
        minmed = min(minmed, gmedian)
        minmode = min(minmode, gmode)
        maxmean = max(maxmean, gmean)
        maxmed = max(maxmed, gmedian)
        maxmode = max(maxmode, gmode)
    axs[0].plot(gval_s[min_g:max_g], np.array(gmeans), label=f"RG Step {step}")
    axs[1].plot(gval_s[min_g:max_g], np.array(gmedians), label=f"RG Step {step}")
    axs[2].plot(gval_s[min_g:max_g], np.array(gmodes), label=f"RG Step {step}")

metrics = [r"$g_{mean}$", r"$g_{median}$", r"$g_{mode}$"]
axlabels = [r"$g_{init}$", r"$g_{metric}$"]
if var != "g":
    metricss = [m.replace("g", var) for m in metrics]
    labelss = [l.replace("g", var) for l in axlabels]
else:
    metricss = metrics
    labelss = axlabels
minmetrics = [minmean*1.1, minmed*1.1, minmode*1.1]
maxmetrics = [maxmean*1.1, maxmed*1.1, maxmode*1.1]
fig.text(0.5, 0.04, labelss[0])
fig.text(0.08, 0.5, labelss[1])
thval = theta_fixed[0]/np.pi
plt.suptitle(f"Plot of {var} metrics after {nsteps} RG steps for theta = {thval}π and {nsamples} samples")
# axs[0].legend(loc="upper left")
for i, ax in enumerate(axs):
    ax.set_xticks(gval_s[min_g:max_g:4])
    ax.set_title(metricss[i])
    if var == "z":
        ax.legend(loc="upper left")
        ax.set_ylim((max(-25, minmetrics[i]), min(15, maxmetrics[i])))
    elif var == "x":
        ax.legend(loc="upper left")
        ax.set_ylim((max(-25, minmetrics[i]), min(10, maxmetrics[i])))
    else:
        ax.legend(loc="upper left")
    # ax.set_xlim((0.8, 0.99))
# plt.xlabel(r"$g_{init}$")
# plt.ylabel(r"g_{mean}")
# plt.xlim((min_g/1.2, max_g*1.2))
# plt.yticks(g_inits)
# plt.xticks(theta_fixed[::2], labels=[str(t) for t in theta_fixed[::2]/(np.pi/2)])
# plt.xticks(g_s[min_g:max_g:3])
# plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))
# plt.title(f"Plot of g metrics after {nsteps} RG steps for theta = {theta_fixed[0]} and {nsamples} samples")
# plt.grid(alpha=0.5)
# plt.tight_layout()
fig.savefig(f"{var}_metrics_theta_{thval:.2f}π_{nsamples}_samples.png")
plt.show()

### Some newerrr trials. Set g = $t^2$, $r^2 + f^2 + t^2 = 1$, Spin-mixing param = q $\in$ [0, 1], $f^2$ = (1-g)*q, $r^2$ = 1 - g - $f^2$

#### Helper funcs for q-parametrisation trials

In [ ]:
def quicker_trials(qs, phis, g_init, samples, nsteps, metric = "mean", var = "g", fix = True):
    gmeans = defaultdict(list)
    qmeds = defaultdict(list)
    for q in qs:
        t_init = rng.uniform(np.sqrt(g_init-1e-5), np.sqrt(g_init+1e-5), samples)
        f_init = q*(1 - t_init**2)
        # print(np.mean(t_init), np.mean(f_init), g_init)
        for step in range(nsteps):
            indices=  rng.integers(0, samples, (samples, 5))
            ts = np.take(t_init, indices)
            fs = np.take(f_init, indices)
            tp = numerical_solver(ts, fs, phis, samples, 2, [1.0,0.0,0.0,0.0], samples)
            tp = np.clip(tp, 1e-6, 1.0-1e-6)
            p = tp**2
            if not fix:
                fp = numerical_solver(ts, fs, phis, samples, 17, [1.0,0.0,0.0,0.0], samples)
            else:
                fp = np.sqrt((1-p)*q)
            # fp = np.clip(fp, 1e-6, 1.0-1e-6)
            f2 = fp**2
            g = p + f2
            tmax = np.max(tp)
            tmin = np.min(tp)
            fmax = np.max(fp)
            fmin = np.min(fp)
            if tmax > 1.0 or fmax > 1.0 or np.max(f2) > 1.0:
                print(f"tmax = {tmax}, fmax = {fmax}. At step {step} for ginit = {g_init} and q = {q}")
            if tmin < 0.0 or fmin < 0.0 or np.min(f2) < 0.0:
                print(f"tmax = {tmax}, fmax = {fmax}. At step {step} for ginit = {g_init} and q = {q}")
            # g = np.clip(g, 1e-12, 1.0-1e-12)
            # tp= np.clip(tp, 1e-12, 1-1e-12)
            # fp = np.clip(fp, 1e-12, 1-1e12)
            if var == "z":
                gp = convert_g_to_z(g)
                # gp = np.log((1 - g - f2)/g)
            elif var == "x":
                gp = convert_g_to_x(g)
            elif var == "p":
                gp = p
            else:
                gp = g
            unique, counts = np.unique(gp, return_counts=True)
            maxunique = np.argmax(counts)
            mode = unique[maxunique]
            median = np.median(gp)
            mean = np.mean(gp)
            qmed = np.median(f2/(1-p))
            qmean = np.mean(f2/(1-p))
            qmeds[step].append((qmean, qmed))
            if metric == "mode":
                gmeans[step].append(mode)
            elif metric == "median":
                gmeans[step].append(median)
            elif metric == "all":
                gmeans[step].append((mean, median))
            else:
                gmeans[step].append(mean)
            t_init = tp
            f_init = fp
    # print(q, qmeds)
    return gmeans, qmeds

In [ ]:
# view_step = 0
# plt.figure(figsize=(12, 8))
plt.ioff()
def plot_data(g_s, q_fixed, nsteps, gms, n, var = "g", test_dir = qshe_test_dir):
    fig, axs = plt.subplots(1, 2, figsize=(16, 8))
    min_g = 0
    max_g = len(g_s)
    minmean,minmed = 0,0
    maxmean,maxmed = 0,0
    gmetrics = {}
    if var == "z":
        # gval_s  = convert_g_to_z(g_s)
        gval_s = np.log(g_s)
    elif var == "x":
        gval_s = convert_g_to_x(g_s, q_fixed[0])
    else:
        gval_s = g_s
    for step in range(nsteps):
        gmeans = []
        gmedians = []
        for i, ginit in enumerate(g_s[min_g:max_g]):
            gmean, gmedian = gms[ginit][step][0]
            if var == "x":
                gmean = convert_g_to_x(gmean, q_fixed[0])
                gmedian = convert_g_to_x(gmedian, q_fixed[0])
            elif var == "z":
                # gmean = convert_g_to_zs(gmean, q_fixed[0])
                # gmedian = convert_g_to_zs(gmedian, q_fixed[0])
                # gmode = convert_g_to_zs(gmode, q_fixed[0])
                gmean = convert_g_to_z(gmean)
                gmedian = convert_g_to_z(gmedian)
            gmeans.append(gmean)
            gmedians.append(gmedian)
            minmean = min(minmean, gmean)
            minmed = min(minmed, gmedian)
            maxmean = max(maxmean, gmean)
            maxmed = max(maxmed, gmedian)
        axs[0].plot(gval_s[min_g:max_g], np.array(gmeans), label=f"RG Step {step}")
        axs[1].plot(gval_s[min_g:max_g], np.array(gmedians), label=f"RG Step {step}")
        # axs[0].plot(gval_s[min_g:max_g], np.array(gmeans), alpha=0.3)
        # axs[1].plot(gval_s[min_g:max_g], np.array(gmedians), alpha=0.3)
        gmetrics.update({step: {"Mean": gmeans, "Median": gmedians}})


    metrics = [r"$g_{mean}$", r"$g_{median}$"]
    axlabels = [r"$g_{init}$", r"$g_{metric}$"]
    if var != "g":
        metricss = [m.replace("g", var) for m in metrics]
        labelss = [l.replace("g", var) for l in axlabels]
    else:
        metricss = metrics
        labelss = axlabels
    minmetrics = [minmean*1.1, minmed*1.1]
    maxmetrics = [maxmean*1.1, maxmed*1.1]
    fig.text(0.5, 0.04, labelss[0])
    fig.text(0.08, 0.5, labelss[1])
    thval = q_fixed[0]
    plt.suptitle(f"Plot of {var} metrics after {nsteps} RG steps for q = {thval:.3f} and {n} samples")
    # axs[0].legend(loc="upper left")
    tick_step = max_g // 10
    for i, ax in enumerate(axs):
        ax.set_xticks(np.round(gval_s[min_g:max_g:tick_step], 3))
        ax.set_title(metricss[i])
        ax.grid()
        if var == "z":
            ax.legend(loc="lower left")
            ax.set_ylim((max(-25, minmetrics[i]), min(15, maxmetrics[i])))
        elif var == "x":
            ax.legend(loc="lower right")
            ax.set_ylim((max(-25, minmetrics[i]), min(10, maxmetrics[i])))
        else:
            ax.legend(loc="upper left")
            ax.set_ylim((-0.05, 1.05))
            ax.set_yticks(np.linspace(0, 1, 21))
    os.makedirs(f"{test_dir}/q_{var}", exist_ok=True)
    fig.savefig(f"{test_dir}/q_{var}/{var}_metrics_q_{thval:.2f}_{n}_samples.png")
    return gmetrics
    # plt.show()

In [ ]:
close_old_plots()

#### Actual trials

In [6]:
ntheta = 1
num_g = 200
lower_g = 0.01
upper_g = 0.99
nsamples = 10000
nsteps = 6
c_indices = rng.integers(0, avail_colors, num_g)
m_indices = rng.integers(0, avail_markers, num_g)
colors = np.take(list(loaded_colors.keys()), c_indices)
marks = np.take(markers, m_indices)
var = "p"
fixed = False

In [ ]:
cur_date = get_current_date("day")
cur_test_dir = f"{qshe_test_dir}/{cur_date}/{nsamples}_samples"
qvals = np.linspace(0.0, 0.5, 21)
pvars = [var, "z", "q"]
# qvals = [0.0]
vardata = {float(qval): {} for qval in qvals}
gqs = {}
qmetrics = {}
g_s = np.linspace(lower_g, upper_g, num_g)
store_data = {q: {} for q in qvals}
for q in qvals:
    q_fixed = np.full((ntheta), fill_value = q)
    gms = {g_init: defaultdict(list) for g_init in g_s}
    qms = {g_init: defaultdict(list) for g_init in g_s}
    for i,ginit in enumerate(g_s):
        phis = generate_random_phases(nsamples, rng, 16)
        gms[ginit], qms[ginit] = quicker_trials(q_fixed, phis, ginit, nsamples, nsteps, "all", var, fixed)
        # if i % 5 == 0:
            # print(f"Finished data gen for g_init = {ginit}")
    qmetrics.update({q: qms})
    gqs.update({q: gms})
    print(f"Finished data gen for q = {q}")
    for pvar in pvars:
        if pvar == "q":
            var_data = plot_data(g_s, q_fixed, nsteps, qms, nsamples, pvar,cur_test_dir)
        else:
            var_data = plot_data(g_s, q_fixed, nsteps, gms, nsamples, pvar,cur_test_dir)
        store_data[q].update({pvar: var_data})
    vardata[q] = find_intersections(g_s, gms, nsteps)
    close_old_plots()
trial_data = collapse_data(store_data)
data_file = f"{cur_test_dir}/metrics_{nsamples}samples_{nsteps}steps.json"
state_file = f"{cur_test_dir}/metrics_{nsamples}samples_{nsteps}steps_state.json"
state_dict = build_state_dict(qvals, g_s, nsamples, nsteps, "all", fixed)
save_metric_json(trial_data, data_file)
save_metric_json(state_dict, state_file)

In [ ]:
# Separate plot only loop in case existing plotter wasn't satisfactory. Requires gqs data to already be loaded in the jupyter env.
for q in qvals[:]:
    data = gqs[q]
    for pvar in pvars:
        if pvar == "q":
            data = qmetrics[q]
            plot_data(g_s, np.array([q]), nsteps, data, pvar, cur_test_dir)
        else:
            plot_data(g_s, np.array([q]), nsteps, data, pvar, cur_test_dir)
        close_old_plots()

### Analysis of gq data

In [ ]:
# Lets try to find the point where the shift between subsequent RG steps change
# Pick two consecutive RG steps to analyse
def get_meds(step_a, step_b, data_dict, qarray, garray, var = "p", fromjson = False):
    medgs_a = {}
    medgs_b = {}
    meangs_a = {}
    meangs_b = {}
    for q in qarray[:]:
        # Other data extraction method if we load the json data instead
        if fromjson:
            gs = data_dict[f"{q}"][var]
            step_a_meds = gs[f"{step_a}"]["Median"]
            step_b_meds = gs[f"{step_b}"]["Median"]
            step_a_means = gs[f"{step_a}"]["Mean"]
            step_b_means = gs[f"{step_b}"]["Mean"]
        else:
            # Otherwise, we want to generate arrays of the mean/median for all ginits, for two consecutive RG steps.
            gs = data_dict[q]
            step_a_meds = []
            step_b_meds = []
            step_a_means = []
            step_b_means = []
            for ginit in garray:
                meang_a, medg_a = gs[ginit][step_a][0]
                meang_b, medg_b = gs[ginit][step_b][0]
                step_a_meds.append(medg_a)
                step_b_meds.append(medg_b)
                step_a_means.append(meang_a)
                step_b_means.append(meang_b)

        medgs_a.update({q:np.array(step_a_meds)})
        medgs_b.update({q:np.array(step_b_meds)})
        meangs_a.update({q:np.array(step_a_means)})
        meangs_b.update({q:np.array(step_b_means)})
    return medgs_a, medgs_b, meangs_a, meangs_b

def plot_delta_med(gmeds_a, gmeds_b, qarray, garray, step1, step2, var, metrictype, plotfolder, qstep, n, save = True):
    close_old_plots()
    ylab= f"$\\Delta {{{var}}}_{{{metrictype}}}$"
    xlab = f"${{{var}}}_{{init}}$"
    plt.figure(figsize=(12, 8))
    plt.axhline(y=0.0, linestyle="--", color="r", alpha=0.4)
    deltas = {q:[] for q in qarray}
    for q in qarray[::qstep]:
        crossover_index = 0
        deltag_mean = gmeds_b[q] - gmeds_a[q]
        # deltag_mean = meangs_b[q] - meangs_a[q]
        for i, j in enumerate(deltag_mean[::]):
            if i+1 >= len(deltag_mean):
                continue
            else:
                if deltag_mean[i+1] > 0.0 and deltag_mean[i] < 0.0 and deltag_mean[i+1] - deltag_mean[i] > 5e-3:
                    # print(deltag_med[i+1] - deltag_med[i])
                    # print(f"Changing index for q = {q:.3f} from {crossover_index} to {i}; -ve to +ve change.")
                    crossover_index = i+1
                elif deltag_mean[i+1] < 0.0 and deltag_mean[i] > 0.0 and np.abs(deltag_mean[i+1] - deltag_mean[i]) > 5e-3:
                    # print(deltag_med[i+1] - deltag_med[i])
                    # print(f"Changing index for q = {q:.3f} from {crossover_index} to {i}; +ve to -ve change.")
                    crossover_index = i+1
        # print(crossover_index)
        a=plt.plot(garray[::], deltag_mean, label=f"q = {q:.3f}")
        deltas[q] = deltag_mean
        # c=a[0].get_color()
        # plt.fill_between(g_s, 0.0, deltag_med, alpha=0.2, color=c)
        # plt.axvline(x=g_s[crossover_index], linestyle="--", alpha=0.5, color=c, label=f"q = {q:.3f}, g[{crossover_index}] = {g_s[crossover_index]:.3f}")

    plt.xticks(garray[::15])
    plt.grid(alpha=0.6)
    plt.ylabel(ylab)
    plt.xlabel(xlab)
    # plt.ylim((-0.01, 0.01))
    plt.title(f"{ylab} vs {xlab} for RG steps {step1},{step2} with {n} samples")
    plt.legend(loc = "upper right", bbox_to_anchor = (1.25, 1.0))
    if save:
        os.makedirs(f"{plotfolder}/plots", exist_ok=True)
        plt.savefig(f"{plotfolder}/plots/{var}_{metrictype}_RG{step1}-{step2}.png", dpi  = 150, bbox_inches="tight")
    return deltas
    # plt.show()

In [ ]:
# step_a = 0
nsamp = nsamples
# nsamp = 50000
mvar = "q"
fromjson = False
mets = ["mean", "med"]
jsonfile = f"{cur_test_dir}/metrics_{nsamp}samples_{nsteps}steps.json"
for met in mets:
    for step_a in range(nsteps-1):
        if fromjson:
            # print("Loading data from json file")
            with open(jsonfile, "r") as f:
                data = json.load(f)
        else:
            if mvar == "q":
                data = qmetrics
            else:
                data = gqs
        step_b = step_a + 1
        meda, medb, meana, meanb = get_meds(step_a, step_b, data, qvals, g_s, var=mvar, fromjson=fromjson)
        if met == "med":
            dat_a = meda
            dat_b = medb
        elif met == "mean":
            dat_a = meana
            dat_b = meanb
        dat = plot_delta_med(dat_a, dat_b, qvals, g_s, step_a, step_b, mvar, met, cur_test_dir, 3, nsamp)

In [ ]:
close_old_plots()
view_step = 1
plt.figure(figsize=(12, 8))
min_g = 0
max_g = num_g
# xvals = convert_g_to_x(g_inits[min_g:max_g])
for i, ginit in enumerate(g_s[min_g:max_g:20]):
    plt.axvline(x=ginit, linestyle="--", color = colors[i+1], label=f"$p_{{init}}$ = {ginit:.3f}", alpha=0.6)

    gvals = []
    for q in qvals[:-2]:
        gs = gqs[q]
        mean, med = gs[ginit][view_step][0]
        gvals.append(mean)
        plt.scatter(mean, q, color=colors[i+1], marker=marks[i])
    plt.fill_betweenx(qvals[:-2], ginit, gvals, color=colors[i+1], alpha=0.2)

# plt.axvline(0.0, linestyle="--", alpha=0.4, label=r"$\theta = 0$", color="m")
# plt.axvline(np.pi, linestyle="--", alpha=0.4, label=r"$\theta=\pi$", color="m")
plt.ylabel(r"$q$")
plt.xlabel(r"$p_{init}$")
# plt.xlim((min_g/1.2, max_g*1.2))
plt.title(f"Plot of $p_{{mean}}$ after {view_step+1} RG step for {len(qvals)-2} qs and {nsamples} samples")
plt.yticks(qvals[:-2], labels=[str(round(float(q), 3)) for q in qvals[:-2]])
# plt.yticks(thetas[::2], labels=[str(t) for t in thetas[::2]/(np.pi)])
plt.xticks(g_s[min_g:max_g:20])
# plt.xlim((g_inits[max(min_g-1, 0)], g_inits[min(max_g+1, num_g-1)]))
plt.legend(loc="upper right", bbox_to_anchor=(1.2, 1.0))
plt.grid(alpha=0.5)
plt.show()
# os.makedirs(f"{qshe_test_dir}/Theta_G/{nsamples}samples/{view_step+1}steps", exist_ok=True)
# plt.savefig(f"{qshe_test_dir}/Theta_G/{nsamples}samples/{view_step+1}steps/theta_g{g_inits[min_g]:.2f}-{g_inits[max_g]:.2f}_{ntheta}θ_{nsamples}samples_{view_step+1}steps.png", dpi=150, bbox_inches="tight")

### Testing rg functions in utilities.py

In [7]:
ntheta = 1
num_g = 500
num_q = 51
lower_g = 0.01
upper_g = 0.99
nsamples = 50000
nsteps = 9
c_indices = rng.integers(0, avail_colors, num_g)
m_indices = rng.integers(0, avail_markers, num_g)
colors = np.take(list(loaded_colors.keys()), c_indices)
marks = np.take(markers, m_indices)
var = "p"
fixed = 0

In [8]:
qvals = np.linspace(0.0, 0.5, num_q)
gvals = np.linspace(lower_g, upper_g, num_g)

In [ ]:
cur_date = get_current_date("day")
cur_test_dir = f"{qshe_test_dir}/{cur_date}/{nsamples}_samples"
os.makedirs(cur_test_dir, exist_ok=True)
pdata, qdata = run_qp_trials(qvals, gvals, nsamples, nsteps, rng, fixed=fixed,output_vars=["t", "f"])
np.save(f"{cur_test_dir}/p_data.npy", pdata)
np.save(f"{cur_test_dir}/q_data.npy", qdata)
trial_state = build_state_dict(qvals, gvals, nsamples, nsteps, "all", fixed)
save_metric_json(trial_state, f"{cur_test_dir}/trial_state_{nsamples}_samples.json")

In [ ]:
# print(pdata.shape)
# print(pdata.size)
# print(pdata)
pval = 100
qval = 0
for i in range(nsteps):
    # plt.scatter(qvals, pdata[:, pval, i, 0])
    plt.plot(gvals, pdata[qval, :, i, 0], label=f"Step {i}")
    # plt.plot(gvals, pdata[0, :, i, 0])
plt.legend(loc="upper right",bbox_to_anchor=(1.25, 1.0))
plt.title(f"$p_{{mean}}$ vs $p_{{init}}$ for $q_{{init}}$ = {qvals[qval]:.3f} for {nsteps} RG steps")
plt.show()
# print(pdata[1,1,:,1])